# LSMC Duration Sweep

Diagnostic notebook: execute `04_lsmc_valuation.ipynb` for durations from `0.5h` to `4.0h` in `0.5h` steps, then plot the LSMC annual mean. This temporarily widens `src.config.VALID_ASSET_DURATIONS_H` inside each executed notebook so the production config file is not edited.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
SOURCE_NOTEBOOK = NOTEBOOK_DIR / "04_lsmc_valuation.ipynb"

SWEEP_DURATIONS_H = [x / 2 for x in range(1, 9)]  # 0.5, 1.0, ..., 4.0
SWEEP_ALLOWED_DURATIONS = tuple(SWEEP_DURATIONS_H)
PHASE4_RUN_MODE_FOR_SWEEP = "medium"  # use "partial" only for a quick smoke test
SWEEP_SEEDS = [42, 123, 456]  # run each (scenario, duration) under multiple seeds
SWEEP_SCENARIOS = {
    "energy_only": {"dc_levels": [0.0], "qr_levels": [0.0]},
    # Cooptimised forward simulation is the memory-heavy case; one worker is slower but steadier on Windows.
    "cooptimised": {"fwd_workers": 1},
}
SKIP_COMPLETED_EXECUTIONS = True  # resume long sweeps without rerunning successful notebooks
START_AT_SCENARIO = globals().get("START_AT_SCENARIO", None)  # optional manual restart point
START_AT_DURATION_H = globals().get("START_AT_DURATION_H", None)
START_AT_SEED = globals().get("START_AT_SEED", None)
RESUME_OUTPUT_DIR = globals().get("RESUME_OUTPUT_DIR", None)
OUTPUT_DIR = Path(RESUME_OUTPUT_DIR) if RESUME_OUTPUT_DIR else Path(globals().get("OUTPUT_DIR", PROCESSED / "notebook_runs" / f"lsmc_duration_sweep_{datetime.now():%Y%m%d_%H%M%S}"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Sweep output dir: {OUTPUT_DIR}")
print(f"Durations: {SWEEP_DURATIONS_H}")
print(f"Scenarios: {list(SWEEP_SCENARIOS)}")
print(f"Seeds: {SWEEP_SEEDS}")
print(f"Phase 4 run mode: {PHASE4_RUN_MODE_FOR_SWEEP}")
print(f"Skip completed executions: {SKIP_COMPLETED_EXECUTIONS}")
if START_AT_SCENARIO:
    print(f"Manual restart point: {START_AT_SCENARIO} / {float(START_AT_DURATION_H):g}h / seed={START_AT_SEED}")
else:
    print("Manual restart point: none; scan from beginning and skip finished runs")


In [ ]:
def duration_label(duration_h: float) -> str:
    return f"{duration_h:g}h"


def run_label(scenario_name: str, duration_h: float, seed: int) -> str:
    return f"{scenario_name}_{duration_label(duration_h)}_seed{seed}"


def backup_existing_outputs(duration_h: float) -> None:
    """Keep a copy of existing duration-labelled Phase 4 outputs before the sweep overwrites them."""
    label = duration_label(duration_h)
    backup_dir = OUTPUT_DIR / "pre_sweep_backups"
    backup_dir.mkdir(exist_ok=True)
    for suffix in ("json", "png", "pkl"):
        for path in PROCESSED.glob(f"lsmc_valuation*_{label}.{suffix}"):
            shutil.copy2(path, backup_dir / path.name)


def parameterized_notebook(src_path: Path, duration_h: float, scenario_name: str, overrides: dict, seed: int, work_dir: Path) -> Path:
    """Create a temporary notebook with duration, seed, and config overrides injected as the first cell."""
    nb = json.loads(src_path.read_text(encoding="utf-8"))
    label = duration_label(duration_h)
    rl = run_label(scenario_name, duration_h, seed)
    scenario_dir = work_dir / scenario_name
    scenario_dir.mkdir(exist_ok=True)
    param_cell = {
        "cell_type": "code",
        "execution_count": None,
        "id": f"duration-sweep-params-{rl.replace('.', '-')}",
        "metadata": {"tags": ["parameters"]},
        "outputs": [],
        "source": [
            "import src.config as _sweep_config\n",
            f"_sweep_config.VALID_ASSET_DURATIONS_H = {SWEEP_ALLOWED_DURATIONS!r}\n",
            f"VALUATION_DURATION_H = {duration_h!r}\n",
            f"PHASE4_RUN_MODE = {PHASE4_RUN_MODE_FOR_SWEEP!r}\n",
            f"PHASE4_MODE_OVERRIDES = {overrides!r}\n",
            f"SEED = {seed!r}\n",
            f"print('Injected LSMC duration sweep: {label}')\n",
            f"print('Injected sweep scenario: {scenario_name}')\n",
            f"print('Injected Phase 4 run mode: {PHASE4_RUN_MODE_FOR_SWEEP}')\n",
            f"print('Injected seed: {seed}')\n",
            "print('Injected Phase 4 overrides:', PHASE4_MODE_OVERRIDES)\n",
        ],
    }
    nb["cells"] = [param_cell, *nb.get("cells", [])]
    tmp_path = scenario_dir / f"{src_path.stem}_{rl}.parameterized.ipynb"
    tmp_path.write_text(json.dumps(nb, indent=1), encoding="utf-8")
    return tmp_path


In [ ]:
if "duration_label" not in globals():
    def duration_label(duration_h: float) -> str:
        return f"{duration_h:g}h"

if "run_label" not in globals():
    def run_label(scenario_name: str, duration_h: float, seed: int) -> str:
        return f"{scenario_name}_{duration_label(duration_h)}_seed{seed}"

if "backup_existing_outputs" not in globals():
    def backup_existing_outputs(duration_h: float) -> None:
        label = duration_label(duration_h)
        backup_dir = OUTPUT_DIR / "pre_sweep_backups"
        backup_dir.mkdir(exist_ok=True)
        for suffix in ("json", "png", "pkl"):
            for path in PROCESSED.glob(f"lsmc_valuation*_{label}.{suffix}"):
                shutil.copy2(path, backup_dir / path.name)

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

scenario_order = list(SWEEP_SCENARIOS)
manual_restart_enabled = (
    START_AT_SCENARIO is not None
    and START_AT_DURATION_H is not None
    and START_AT_SEED is not None
)
start_scenario_index = scenario_order.index(START_AT_SCENARIO) if manual_restart_enabled else 0
start_duration_h = float(START_AT_DURATION_H) if manual_restart_enabled else min(SWEEP_DURATIONS_H)
start_seed = int(START_AT_SEED) if manual_restart_enabled else SWEEP_SEEDS[0]

def is_before_restart_point(scenario_name: str, duration_h: float, seed: int) -> bool:
    if not manual_restart_enabled:
        return False
    si = scenario_order.index(scenario_name)
    if si < start_scenario_index:
        return True
    if si == start_scenario_index:
        if duration_h < start_duration_h:
            return True
        if duration_h == start_duration_h and seed < start_seed:
            return True
    return False

completed_from_prior_summary = set()
prior_summary_path = OUTPUT_DIR / "lsmc_duration_sweep_run_summary.csv"
if prior_summary_path.exists():
    prior_summary = pd.read_csv(prior_summary_path)
    for row in prior_summary.to_dict("records"):
        if int(row.get("returncode", 1)) == 0:
            completed_from_prior_summary.add((row.get("scenario"), row.get("duration_label"), int(row.get("seed", 42))))

run_rows = []

for scenario_name, overrides in SWEEP_SCENARIOS.items():
    scenario_dir = OUTPUT_DIR / scenario_name
    scenario_dir.mkdir(exist_ok=True)

    for duration_h in SWEEP_DURATIONS_H:
        label = duration_label(duration_h)
        backup_existing_outputs(duration_h)

        for seed in SWEEP_SEEDS:
            rl = run_label(scenario_name, duration_h, seed)
            out_name = f"{SOURCE_NOTEBOOK.stem}_{rl}.executed.ipynb"
            out_path = scenario_dir / out_name
            summary_copy = scenario_dir / f"lsmc_valuation_summary_{rl}.json"
            completed = (out_path.exists() and summary_copy.exists()) or (
                (scenario_name, label, seed) in completed_from_prior_summary and summary_copy.exists()
            )
            if is_before_restart_point(scenario_name, duration_h, seed):
                print(f"Skipping {scenario_name} {label} seed={seed}; before restart point")
                run_rows.append({
                    "scenario": scenario_name, "duration_h": duration_h,
                    "duration_label": label, "seed": seed,
                    "returncode": 0 if completed else None, "elapsed_s": 0.0,
                    "executed_notebook": out_name, "summary_json": summary_copy.name,
                    "skipped": True, "skip_reason": "before_restart_point",
                })
                continue
            if SKIP_COMPLETED_EXECUTIONS and completed:
                print(f"Skipping {scenario_name} {label} seed={seed}; already finished")
                run_rows.append({
                    "scenario": scenario_name, "duration_h": duration_h,
                    "duration_label": label, "seed": seed,
                    "returncode": 0, "elapsed_s": 0.0,
                    "executed_notebook": out_name, "summary_json": summary_copy.name,
                    "skipped": True, "skip_reason": "completed",
                })
                continue

            tmp_src = parameterized_notebook(SOURCE_NOTEBOOK, duration_h, scenario_name, overrides, seed, OUTPUT_DIR)
            cmd = [
                sys.executable, "-m", "jupyter", "nbconvert",
                "--to", "notebook", "--execute",
                "--ExecutePreprocessor.timeout=-1",
                "--ExecutePreprocessor.kernel_name=python3",
                f"--output-dir={scenario_dir}",
                f"--output={out_name}",
                str(tmp_src),
            ]

            print("\n" + "=" * 80)
            print(f"Executing {SOURCE_NOTEBOOK.name} for {scenario_name} / {label} / seed={seed}")
            print("=" * 80)
            started = time.time()
            proc = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, text=True,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            elapsed = time.time() - started
            print(proc.stdout[-4000:])
            print(f"Finished {scenario_name} / {label} / seed={seed} in {elapsed:.1f}s with return code {proc.returncode}")
            if proc.returncode == 0:
                produced_summary = PROCESSED / f"lsmc_valuation_summary_{label}.json"
                if produced_summary.exists():
                    shutil.copy2(produced_summary, summary_copy)
                produced_plot = PROCESSED / f"lsmc_valuation_{label}.png"
                if produced_plot.exists():
                    shutil.copy2(produced_plot, scenario_dir / f"lsmc_valuation_{rl}.png")

            run_rows.append({
                "scenario": scenario_name, "duration_h": duration_h,
                "duration_label": label, "seed": seed,
                "returncode": proc.returncode, "elapsed_s": elapsed,
                "executed_notebook": out_name, "summary_json": summary_copy.name,
                "skipped": False,
            })
            pd.DataFrame(run_rows).to_csv(OUTPUT_DIR / "lsmc_duration_sweep_run_summary.csv", index=False)
            if proc.returncode != 0:
                raise RuntimeError(f"04_lsmc_valuation.ipynb failed for {scenario_name} / {label} / seed={seed}")

run_summary = pd.DataFrame(run_rows)
run_summary.to_csv(OUTPUT_DIR / "lsmc_duration_sweep_run_summary.csv", index=False)
run_summary


In [ ]:
if "duration_label" not in globals():
    def duration_label(duration_h: float) -> str:
        return f"{duration_h:g}h"

if "run_label" not in globals():
    def run_label(scenario_name: str, duration_h: float, seed: int) -> str:
        return f"{scenario_name}_{duration_label(duration_h)}_seed{seed}"

summary_rows = []
expected_run_mode = PHASE4_RUN_MODE_FOR_SWEEP
LSMC_RI_RATIO_WARN = 10.0  # flag runs where LSMC/RI ratio exceeds this threshold

for scenario_name in SWEEP_SCENARIOS:
    scenario_dir = OUTPUT_DIR / scenario_name
    for duration_h in SWEEP_DURATIONS_H:
        label = duration_label(duration_h)
        for seed in SWEEP_SEEDS:
            rl = run_label(scenario_name, duration_h, seed)
            path = scenario_dir / f"lsmc_valuation_summary_{rl}.json"
            if not path.exists():
                raise FileNotFoundError(path)
            with path.open("r", encoding="utf-8") as f:
                summary = json.load(f)
            actual_run_mode = summary.get("run_mode")
            if actual_run_mode != expected_run_mode:
                raise RuntimeError(
                    f"{path.name} has run_mode={actual_run_mode!r}, expected {expected_run_mode!r}. "
                    "Rerun the execution cell with SKIP_COMPLETED_EXECUTIONS = False."
                )
            action = summary.get("action_distribution", {})
            diag = summary.get("lsmc_diagnostics", {})
            by_mode = action.get("by_mode", [])
            true_idle_fraction = sum(
                mode.get("fraction", 0.0)
                for mode in by_mode
                if mode.get("net_frac") == 0.0 and mode.get("r_dc_frac") == 0.0 and mode.get("r_qr_frac") == 0.0
            )
            reserve_only_fraction = sum(
                mode.get("fraction", 0.0)
                for mode in by_mode
                if mode.get("net_frac") == 0.0 and (mode.get("r_dc_frac", 0.0) > 0.0 or mode.get("r_qr_frac", 0.0) > 0.0)
            )
            lsmc_ri_ratio = summary.get("lsmc_ri_ratio")
            ratio_flagged = (lsmc_ri_ratio is not None and lsmc_ri_ratio > LSMC_RI_RATIO_WARN)
            if ratio_flagged:
                print(f"  WARNING: {scenario_name} {label} seed={seed} — LSMC/RI ratio={lsmc_ri_ratio:.1f}x exceeds {LSMC_RI_RATIO_WARN:.0f}x threshold")
            summary_rows.append({
                "scenario": scenario_name,
                "duration_h": duration_h,
                "duration_label": label,
                "seed": seed,
                "asset_mwh": summary.get("asset_mwh"),
                "run_mode": summary.get("run_mode"),
                "n_paths": summary.get("n_paths"),
                "n_steps": summary.get("n_steps"),
                "valuation_horizon_years": summary.get("valuation_horizon_years"),
                "annualization_factor": summary.get("annualization_factor"),
                "lsmc_annual_mean_gbp": summary.get("mtm_gbp_annualized", {}).get("mean"),
                "lsmc_gbp_per_mw_year": summary.get("mtm_gbp_per_mw_year", {}).get("mean"),
                "cashflow_mean_gbp": action.get("cashflow_mean_gbp"),
                "charge_fraction": action.get("charge_fraction"),
                "discharge_fraction": action.get("discharge_fraction"),
                "idle_fraction": action.get("idle_fraction"),
                "true_idle_fraction": true_idle_fraction,
                "reserve_only_fraction": reserve_only_fraction,
                "dominant_action_fraction": action.get("dominant_action_fraction"),
                "lsmc_ri_ratio": lsmc_ri_ratio,
                "ratio_flagged": ratio_flagged,
                "beta_abs_max": diag.get("beta_abs_max"),
                "target_std_max": diag.get("target_std_max"),
                "continuation_clip_fraction_max": diag.get("continuation_clip_fraction_max"),
                "continuation_value_cap_gbp": diag.get("continuation_value_cap_gbp"),
            })

sweep_raw = pd.DataFrame(summary_rows)
sweep_raw.to_csv(OUTPUT_DIR / "lsmc_duration_sweep_raw.csv", index=False)

# Aggregate across seeds: mean ± std per (scenario, duration)
agg_cols = ["lsmc_annual_mean_gbp", "lsmc_gbp_per_mw_year", "lsmc_ri_ratio",
            "cashflow_mean_gbp", "beta_abs_max", "target_std_max",
            "charge_fraction", "discharge_fraction", "idle_fraction",
            "true_idle_fraction", "reserve_only_fraction"]
sweep = (
    sweep_raw.groupby(["scenario", "duration_h", "duration_label"])[agg_cols]
    .agg(["mean", "std"])
    .reset_index()
)
sweep.columns = ["scenario", "duration_h", "duration_label"] + [
    f"{col}_{stat}" for col, stat in sweep.columns[3:]
]
sweep["n_seeds"] = sweep_raw.groupby(["scenario", "duration_h"])["seed"].count().values
sweep["ratio_flagged_any"] = (
    sweep_raw.groupby(["scenario", "duration_h"])["ratio_flagged"].any().values
)

sweep.to_csv(PROCESSED / "lsmc_duration_sweep.csv", index=False)
sweep.to_csv(OUTPUT_DIR / "lsmc_duration_sweep.csv", index=False)

print(f"Raw per-seed rows: {len(sweep_raw)}")
print(f"Aggregated rows:   {len(sweep)}")
flagged = sweep_raw[sweep_raw["ratio_flagged"]]
if len(flagged):
    print(f"\nFlagged runs (LSMC/RI > {LSMC_RI_RATIO_WARN:.0f}x):")
    print(flagged[["scenario", "duration_label", "seed", "lsmc_ri_ratio"]].to_string(index=False))
else:
    print("No runs exceeded the LSMC/RI ratio threshold.")
sweep


In [ ]:
import numpy as np

colors = {"energy_only": "steelblue", "cooptimised": "darkorange"}

# ── Main curve: mean ± std across seeds ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for scenario_name, group in sweep.groupby("scenario", sort=False):
    group = group.sort_values("duration_h")
    mean_m = group["lsmc_annual_mean_gbp_mean"] / 1_000_000
    std_m  = group["lsmc_annual_mean_gbp_std"] / 1_000_000
    flagged_mask = group["ratio_flagged_any"].values
    col = colors.get(scenario_name, "grey")
    ax.plot(group["duration_h"], mean_m, marker="o", linewidth=2, color=col, label=scenario_name)
    ax.fill_between(group["duration_h"], mean_m - std_m, mean_m + std_m,
                    alpha=0.20, color=col)
    # Mark flagged points with an X overlay
    if flagged_mask.any():
        ax.scatter(group["duration_h"].values[flagged_mask], mean_m.values[flagged_mask],
                   marker="x", s=120, linewidths=2.5, color="red", zorder=5,
                   label=f"{scenario_name} flagged (ratio>{LSMC_RI_RATIO_WARN:.0f}x)")

ax.set_title(f"LSMC annual mean by battery duration ({PHASE4_RUN_MODE_FOR_SWEEP} mode, {len(SWEEP_SEEDS)} seeds)")
ax.set_xlabel("Battery duration (hours)")
ax.set_ylabel("LSMC annual mean (GBP million/year)")
ax.set_xticks(SWEEP_DURATIONS_H)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

plot_path = PROCESSED / "lsmc_duration_sweep.png"
fig.savefig(plot_path, dpi=160)
fig.savefig(OUTPUT_DIR / "lsmc_duration_sweep.png", dpi=160)
print(f"Saved plot: {plot_path}")
plt.show()

# ── Diagnostics: 4 panels ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
ax0, ax1, ax2, ax3, ax4, ax5 = axes.ravel()

for scenario_name, group in sweep.groupby("scenario", sort=False):
    group = group.sort_values("duration_h")
    col = colors.get(scenario_name, "grey")

    def _plot(ax, col_mean, col_std=None, label_suffix=""):
        y = group[col_mean]
        ax.plot(group["duration_h"], y, marker="o", color=col, linewidth=1.8,
                label=scenario_name + label_suffix)
        if col_std and col_std in group.columns:
            s = group[col_std].fillna(0)
            ax.fill_between(group["duration_h"], y - s, y + s, alpha=0.18, color=col)

    _plot(ax0, "lsmc_annual_mean_gbp_mean", "lsmc_annual_mean_gbp_std")
    _plot(ax1, "cashflow_mean_gbp_mean", "cashflow_mean_gbp_std")
    _plot(ax2, "lsmc_ri_ratio_mean", "lsmc_ri_ratio_std")
    _plot(ax3, "reserve_only_fraction_mean", "reserve_only_fraction_std", " reserve-only")
    _plot(ax3, "true_idle_fraction_mean", "true_idle_fraction_std", " true idle")
    _plot(ax4, "beta_abs_max_mean", "beta_abs_max_std")
    _plot(ax5, "target_std_max_mean", "target_std_max_std")

ax0.set_title("Annual mean"); ax0.set_ylabel("GBP/year")
ax1.set_title("Mean cashflow/HH"); ax1.set_ylabel("GBP")
ax2.set_title("LSMC/RI ratio"); ax2.set_ylabel("ratio")
ax2.axhline(LSMC_RI_RATIO_WARN, color="red", linewidth=1, linestyle="--", label=f"warn={LSMC_RI_RATIO_WARN:.0f}x")
ax3.set_title("Idle / reserve mix"); ax3.set_ylabel("fraction")
ax4.set_title("Beta abs max"); ax4.set_ylabel("£")
ax5.set_title("Target std max"); ax5.set_ylabel("£")

for ax in axes.ravel():
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
for ax in axes[-1, :]:
    ax.set_xlabel("Battery duration (hours)")
    ax.set_xticks(SWEEP_DURATIONS_H)

fig.suptitle(
    f"LSMC duration sweep diagnostics ({PHASE4_RUN_MODE_FOR_SWEEP} mode, "
    f"{len(SWEEP_SEEDS)} seeds ± 1σ shading)",
    y=1.01,
)
fig.tight_layout()
diag_plot_path = PROCESSED / "lsmc_duration_sweep_diagnostics.png"
fig.savefig(diag_plot_path, dpi=160, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "lsmc_duration_sweep_diagnostics.png", dpi=160, bbox_inches="tight")
print(f"Saved diagnostic plot: {diag_plot_path}")
plt.show()

# ── Per-seed scatter (secondary view) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
for scenario_name, group in sweep_raw.groupby("scenario", sort=False):
    col = colors.get(scenario_name, "grey")
    group = group.sort_values("duration_h")
    ax.scatter(group["duration_h"] + (0.04 if scenario_name == "cooptimised" else -0.04),
               group["lsmc_annual_mean_gbp"] / 1_000_000,
               alpha=0.5, s=30, color=col, label=scenario_name)
ax.set_title(f"Per-seed scatter ({len(SWEEP_SEEDS)} seeds)")
ax.set_xlabel("Battery duration (hours)")
ax.set_ylabel("LSMC annual mean (GBP million/year)")
ax.set_xticks(SWEEP_DURATIONS_H)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "lsmc_duration_sweep_per_seed.png", dpi=160)
plt.show()


## Optional Restore

The sweep runs Phase 4 directly, so it overwrites local `lsmc_valuation_*_<duration>` files in `data/processed`. Run the next cell if you want to restore any files that existed before the sweep.

In [9]:
backup_dir = OUTPUT_DIR / "pre_sweep_backups"
if backup_dir.exists():
    restored = []
    for path in backup_dir.glob("lsmc_valuation*"):
        target = PROCESSED / path.name
        shutil.copy2(path, target)
        restored.append(target.name)
    print(f"Restored {len(restored)} files from {backup_dir}")
    restored
else:
    print("No pre-sweep backup directory found for this run.")

Restored 24 files from g:\My Drive\Research\bess_project\data\processed\notebook_runs\lsmc_duration_sweep_20260501_111405\pre_sweep_backups
